***Mount Drive & Install***

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install earthengine-api rasterio -q

import ee
ee.Authenticate()
ee.Initialize(project='heroic-calculus-490406-v3')

import os
BASE = '/content/drive/MyDrive/FloodProject/data/sen1floods11'
os.makedirs(f'{BASE}/dem', exist_ok=True)
os.makedirs(f'{BASE}/dem/mosaics', exist_ok=True)
print('Setup complete.')

Mounted at /content/drive
Setup complete.


In [ ]:
import ee
ee.Initialize(project='heroic-calculus-490406-v3')

# Test Copernicus DEM
try:
    dem = ee.ImageCollection('COPERNICUS/DEM/GLO30').select('DEM').mosaic()
    info = dem.getInfo()
    print('✓ Copernicus DEM as ImageCollection works')
except Exception as e:
    print(f'✗ Copernicus failed: {e}')

# Test SRTM fallback
try:
    dem2 = ee.Image('USGS/SRTMGL1_003').select('elevation')
    info = dem2.getInfo()
    print('✓ SRTM works')
except Exception as e:
    print(f'✗ SRTM failed: {e}')

# Test HAND options
for asset, desc in [
    ('users/gena/GlobalHAND/30m/hand', 'gena HAND'),
    ('users/gena/GlobalHAND/30m-global/hand', 'gena HAND v2'),
]:
    try:
        h = ee.Image(asset)
        h.getInfo()
        print(f'✓ {desc} works')
    except Exception as e:
        print(f'✗ {desc} failed')

# Test MERIT
try:
    merit = ee.ImageCollection('MERIT/Hydro/v07_Mrb01').mosaic().select('hnd')
    merit.getInfo()
    print('✓ MERIT/Hydro works')
except Exception as e:
    print(f'✗ MERIT/Hydro failed')

✓ Copernicus DEM as ImageCollection works
✓ SRTM works
✗ gena HAND failed
✗ gena HAND v2 failed
✗ MERIT/Hydro failed


In [ ]:
import ee
ee.Initialize(project='heroic-calculus-490406-v3')

# Test HAND computation from DEM
try:
    # Step 1: Load DEM as ImageCollection mosaic
    dem = ee.ImageCollection('COPERNICUS/DEM/GLO30').select('DEM').mosaic()

    # Step 2: Identify river/drainage pixels (low elevation + high flow accumulation)
    # Using a small test region over India
    test_region = ee.Geometry.Rectangle([90.0, 24.0, 91.0, 25.0], 'EPSG:4326', False)

    # Step 3: Compute flow accumulation proxy via focal mean difference
    focal_min  = dem.focalMin(radius=5, units='pixels')
    hand_proxy = dem.subtract(focal_min).rename('HAND')

    # Test export
    task = ee.batch.Export.image.toDrive(
        image          = hand_proxy.clip(test_region),
        description    = 'TEST_HAND_proxy',
        folder         = 'gee_test',
        fileNamePrefix = 'TEST_HAND_proxy',
        region         = test_region,
        scale          = 30,
        crs            = 'EPSG:4326',
        maxPixels      = int(1e9)
    )
    task.start()
    print(f'✓ HAND proxy computation works — task submitted: {task.id}')
    print('Check GEE Tasks tab — if it runs, we are good to proceed.')

except Exception as e:
    print(f'✗ Failed: {e}')

✓ HAND proxy computation works — task submitted: SFAI6R62NJDV35M7ZUJWP4O4
Check GEE Tasks tab — if it runs, we are good to proceed.


***Export DEM Mosaics from GEE (6 tasks total)***

Bands: Elevation, Slope, TWI, HAND (Computed from DEM)

1 large GeoTIFF per country → Google Drive folder 'sen1floods11_dem_mosaics'

In [ ]:
import ee
ee.Initialize(project='heroic-calculus-490406-v3')

# ── Country bounding boxes ─────────────────────────────────────
COUNTRIES = {
    'India':     [ 89.0,  21.0,  97.0,  28.0],
    'Pakistan':  [ 66.0,  23.0,  74.0,  32.0],
    'Sri-Lanka': [ 79.5,   5.5,  82.5,  10.0],
    'Mekong':    [100.0,  10.0, 108.0,  22.0],
    'Bolivia':   [-69.0, -20.0, -57.0, -10.0],
    'Colombia':  [-78.0,  -5.0, -66.0,   8.0],
}

# ── DEM + Derivatives ─────────────────────────────────────────
# Copernicus GLO-30 is an ImageCollection — must mosaic first
dem = ee.ImageCollection('COPERNICUS/DEM/GLO30').select('DEM').mosaic()

# Slope in degrees
slope = ee.Terrain.slope(dem).rename('Slope')

# TWI = ln(flow_accumulation / tan(slope))
slope_rad      = slope.multiply(3.14159265 / 180.0)
tan_slope      = slope_rad.tan()
tan_slope_safe = tan_slope.where(tan_slope.lt(0.001), 0.001)
flow_acc       = dem.reduceNeighborhood(
    reducer = ee.Reducer.sum(),
    kernel  = ee.Kernel.circle(radius=5, units='pixels')
)
twi = flow_acc.divide(tan_slope_safe).log().rename('TWI')

# HAND = elevation - local minimum elevation (drainage approximation)
# focalMin with radius=50 pixels captures nearest drainage channel
focal_min = dem.focalMin(radius=50, units='pixels')
hand      = dem.subtract(focal_min).rename('HAND')

# Stack: band1=Elevation, band2=Slope, band3=TWI, band4=HAND
dem_stack = dem.rename('Elevation') \
               .addBands(slope) \
               .addBands(twi) \
               .addBands(hand) \
               .toFloat()

# ── Submit 6 Export Tasks ─────────────────────────────────────
print('Submitting export tasks...\n')
for country, bbox in COUNTRIES.items():
    region = ee.Geometry.Rectangle(bbox, 'EPSG:4326', False)
    task   = ee.batch.Export.image.toDrive(
        image          = dem_stack.clip(region),
        description    = f'{country}_DEM_mosaic',
        folder         = 'sen1floods11_dem_mosaics',
        fileNamePrefix = f'{country}_DEM_mosaic',
        region         = region,
        scale          = 30,
        crs            = 'EPSG:4326',
        fileFormat     = 'GeoTIFF',
        maxPixels      = int(1e13)
    )
    task.start()
    print(f'  ✓ {country:<12}: task submitted ({task.id})')

print('\nAll 6 tasks submitted.')
print('Monitor at https://code.earthengine.google.com → Tasks tab')
print('Typical time: 15-40 mins per country depending on size.')
print('Run Cell 3 periodically to check progress.')

Submitting export tasks...

  ✓ India       : task submitted (TXNKYWFTJ4CEGEJ6WL3RV5NQ)
  ✓ Pakistan    : task submitted (E4NITKOYDHXFNNPHQBM37XUG)
  ✓ Sri-Lanka   : task submitted (5DIEGP6KM5EZEW5HZCK3WUFA)
  ✓ Mekong      : task submitted (OTO3FCDHVUVHXHGFEHQNFUNE)
  ✓ Bolivia     : task submitted (HRGVQXULPSX63RE5FVE5IE6W)
  ✓ Colombia    : task submitted (5MZLZJPPQPU3BNA7NRFCCL3K)

All 6 tasks submitted.
Monitor at https://code.earthengine.google.com → Tasks tab
Typical time: 15-40 mins per country depending on size.
Run Cell 3 periodically to check progress.


In [ ]:
import subprocess
# Forcefully unmount
subprocess.run(['fusermount', '-u', '/content/drive'], capture_output=True)
subprocess.run(['rm', '-rf', '/content/drive'], capture_output=True)

# Now remount
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, glob

dirs_to_check = [
    '/content/drive/MyDrive/sen1floods11_dem_mosaics',
    '/content/drive/MyDrive/FloodProject/data/sen1floods11/dem/mosaics',
    '/content/drive/MyDrive/FloodProject/data/sen1floods11/dem/chips',
]

for d in dirs_to_check:
    files = glob.glob(f'{d}/*.tif')
    if not files:
        print(f'\n{d}: NO FILES')
        continue
    print(f'\n{d}')
    total = 0
    for f in sorted(files):
        gb = os.path.getsize(f) / 1e9
        total += gb
        print(f'  {os.path.basename(f):<45}: {gb:.2f} GB')
    print(f'  TOTAL: {total:.2f} GB')

# Also check overall Drive usage
import shutil
total, used, free = shutil.disk_usage('/content/drive/MyDrive')
print(f'\nDrive: {used/1e9:.1f} GB used / {total/1e9:.1f} GB total / {free/1e9:.1f} GB free')


/content/drive/MyDrive/sen1floods11_dem_mosaics
  Bolivia_DEM_mosaic-0000000000-0000000000.tif : 3.21 GB
  Bolivia_DEM_mosaic-0000000000-0000016384.tif : 3.34 GB
  Bolivia_DEM_mosaic-0000000000-0000032768.tif : 2.39 GB
  Bolivia_DEM_mosaic-0000016384-0000000000.tif : 3.23 GB
  Bolivia_DEM_mosaic-0000016384-0000016384.tif : 3.23 GB
  Bolivia_DEM_mosaic-0000016384-0000032768.tif : 2.37 GB
  Bolivia_DEM_mosaic-0000032768-0000000000.tif : 0.75 GB
  Bolivia_DEM_mosaic-0000032768-0000016384.tif : 0.85 GB
  Bolivia_DEM_mosaic-0000032768-0000032768.tif : 0.60 GB
  Colombia_DEM_mosaic-0000000000-0000000000.tif: 3.12 GB
  Colombia_DEM_mosaic-0000000000-0000016384.tif: 3.29 GB
  Colombia_DEM_mosaic-0000000000-0000032768.tif: 2.43 GB
  Colombia_DEM_mosaic-0000016384-0000000000.tif: 3.35 GB
  Colombia_DEM_mosaic-0000016384-0000016384.tif: 3.36 GB
  Colombia_DEM_mosaic-0000016384-0000032768.tif: 2.46 GB
  Colombia_DEM_mosaic-0000032768-0000000000.tif: 3.16 GB
  Colombia_DEM_mosaic-0000032768-000001

***Monitor Export Progress***

Re-run this cell periodically to check status

In [ ]:
import ee, glob
ee.Authenticate()

ee.Initialize(project='heroic-calculus-490406-v3')

GEE_OUT   = '/content/drive/MyDrive/sen1floods11_dem_mosaics'
COUNTRIES = ['India', 'Pakistan', 'Sri-Lanka', 'Mekong', 'Bolivia', 'Colombia']

# GEE task status
tasks  = ee.batch.Task.list()
states = {}
for t in tasks:
    desc = t.config.get('description', '')
    if 'DEM_mosaic' in desc:
        country = desc.replace('_DEM_mosaic', '')
        states[country] = t.state

print('=== GEE Task Status ===')
all_done = True
for country in COUNTRIES:
    state = states.get(country, 'NOT FOUND')
    icon  = '✓' if state == 'COMPLETED' else ('⏳' if state in ['RUNNING', 'READY'] else '✗')
    print(f'  {country:<12}: {state}  {icon}')
    if state != 'COMPLETED':
        all_done = False

# Files in Drive
print()
print('=== Files in Drive ===')
import os
os.makedirs(GEE_OUT, exist_ok=True)
for country in COUNTRIES:
    files = glob.glob(f'{GEE_OUT}/{country}_DEM_mosaic*.tif')
    print(f'  {country:<12}: {len(files)} file(s)')

if all_done:
    print('\nAll exports complete ✓ — run Cell 4 to move files.')
else:
    print('\nStill running — re-run this cell in a few minutes.')

=== GEE Task Status ===
  India       : COMPLETED  ✓
  Pakistan    : COMPLETED  ✓
  Sri-Lanka   : COMPLETED  ✓
  Mekong      : COMPLETED  ✓
  Bolivia     : COMPLETED  ✓
  Colombia    : COMPLETED  ✓

=== Files in Drive ===
  India       : 0 file(s)
  Pakistan    : 0 file(s)
  Sri-Lanka   : 0 file(s)
  Mekong      : 0 file(s)
  Bolivia     : 0 file(s)
  Colombia    : 0 file(s)

All exports complete ✓ — run Cell 4 to move files.


***Move Mosaics to Correct Folder***

Run AFTER all 6 GEE tasks show COMPLETED in above cell

In [ ]:
import glob, shutil, os

GEE_OUT = '/content/drive/MyDrive/sen1floods11_dem_mosaics'
BASE    = '/content/drive/MyDrive/FloodProject/data/sen1floods11'
DEST    = f'{BASE}/dem/mosaics'
os.makedirs(DEST, exist_ok=True)

files = glob.glob(f'{GEE_OUT}/*.tif')
print(f'Moving {len(files)} mosaic files...')
for f in files:
    shutil.move(f, os.path.join(DEST, os.path.basename(f)))
    print(f'  Moved: {os.path.basename(f)}')

print(f'\nDone. Files in {DEST}:')
for f in sorted(glob.glob(f'{DEST}/*.tif')):
    size = os.path.getsize(f) / 1e6
    print(f'  {os.path.basename(f):<35}: {size:.1f} MB')

Moving 0 mosaic files...

Done. Files in /content/drive/MyDrive/FloodProject/data/sen1floods11/dem/mosaics:


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


***DEM TILING — Extract per-chip DEM from country mosaics***

Resume-safe, RAM-safe, Drive-quota-safe

In [ ]:
!apt-get install -y gdal-bin -q
!pip install rasterio tqdm -q

import os, glob, shutil, subprocess
import numpy as np
import rasterio
from rasterio.windows import from_bounds
from rasterio.enums import Resampling
from tqdm import tqdm

# ── Paths ─────────────────────────────────────────────────────
BASE        = '/content/drive/MyDrive/FloodProject/data/sen1floods11'
MOSAIC_DR   = '/content/drive/MyDrive/sen1floods11_dem_mosaics'
LOCAL_TMP   = '/content/dem_tmp'    # mosaic tiles copied here temporarily
LOCAL_CHIPS = '/content/dem_chips'  # chips written here first
DRIVE_CHIPS = f'{BASE}/dem/chips'   # chips copied here after mosaics deleted

os.makedirs(LOCAL_TMP,   exist_ok=True)
os.makedirs(LOCAL_CHIPS, exist_ok=True)
os.makedirs(DRIVE_CHIPS, exist_ok=True)

# ── Country config ─────────────────────────────────────────────
COUNTRIES = {
    'India':     ['train', 'val'],
    'Pakistan':  ['train', 'val'],
    'Sri-Lanka': ['train', 'val'],
    'Mekong':    ['train', 'val'],
    'Bolivia':   ['train'],
    'Colombia':  ['train'],
}

# ── Core tiling function ───────────────────────────────────────
def tile_chips_from_vrt(s1_files, vrt_path, out_dir, suffix):
    done, skipped, failed = 0, 0, 0
    with rasterio.open(vrt_path) as vrt:
        for s1_path in s1_files:
            chip_id  = os.path.basename(s1_path).replace(f'_{suffix}.tif', '')
            out_path = f'{out_dir}/{chip_id}_DEM.tif'

            # Resume-safe: skip if already done
            if os.path.exists(out_path):
                skipped += 1
                continue

            try:
                with rasterio.open(s1_path) as s1:
                    bounds    = s1.bounds
                    out_h     = s1.height   # 512
                    out_w     = s1.width    # 512
                    transform = s1.transform
                    crs       = s1.crs

                # Check chip is within VRT bounds
                vb = vrt.bounds
                if (bounds.right  < vb.left  or
                    bounds.left   > vb.right or
                    bounds.top    < vb.bottom or
                    bounds.bottom > vb.top):
                    failed += 1
                    continue

                window = from_bounds(
                    bounds.left, bounds.bottom,
                    bounds.right, bounds.top,
                    vrt.transform
                )

                # Read only this chip's window — no full mosaic in RAM
                data = vrt.read(
                    window     = window,
                    out_shape  = (vrt.count, out_h, out_w),
                    resampling = Resampling.bilinear
                ).astype(np.float32)

                # Handle nodata
                data = np.where(np.isfinite(data), data, 0.0)

                # Write chip
                profile = {
                    'driver'   : 'GTiff',
                    'dtype'    : 'float32',
                    'width'    : out_w,
                    'height'   : out_h,
                    'count'    : vrt.count,
                    'crs'      : crs,
                    'transform': transform,
                    'compress' : 'lzw',
                }
                with rasterio.open(out_path, 'w', **profile) as dst:
                    dst.write(data)
                done += 1

            except Exception as e:
                print(f'\n  FAILED: {chip_id} — {e}')
                failed += 1

    return done, skipped, failed

# ── PHASE 1: Tile all chips to LOCAL disk ─────────────────────
print('=' * 55)
print('PHASE 1: Tiling chips to Colab local disk')
print('=' * 55)

total_done = 0

for country, splits in COUNTRIES.items():
    print(f'\n{country}...')

    # Find all mosaic tiles for this country
    drive_tiles = sorted(glob.glob(f'{MOSAIC_DR}/{country}_DEM_mosaic*.tif'))
    if not drive_tiles:
        print(f'  WARNING: No mosaic tiles found — skipping')
        continue
    print(f'  Found {len(drive_tiles)} mosaic tiles')

    # Copy mosaic tiles to local disk
    print(f'  Copying to local disk...')
    local_tiles = []
    for t in drive_tiles:
        dst = f'{LOCAL_TMP}/{os.path.basename(t)}'
        if not os.path.exists(dst):
            shutil.copy2(t, dst)
        local_tiles.append(dst)
    local_gb = sum(os.path.getsize(f) for f in local_tiles) / 1e9
    print(f'  Local size: {local_gb:.1f} GB')

    # Build VRT from local tiles (no RAM used)
    vrt_path       = f'{LOCAL_TMP}/{country}.vrt'
    tile_list_path = f'{LOCAL_TMP}/{country}_tiles.txt'
    with open(tile_list_path, 'w') as f:
        f.write('\n'.join(local_tiles))
    result = subprocess.run(
        ['gdalbuildvrt', '-input_file_list', tile_list_path, vrt_path],
        capture_output=True, text=True
    )
    if not os.path.exists(vrt_path):
        print(f'  ERROR: VRT build failed — {result.stderr}')
        continue
    print(f'  VRT built successfully')

    # Tile train chips
    if 'train' in splits:
        train_files = sorted(glob.glob(
            f'{BASE}/train/S1Weak/{country}_*_S1Weak.tif'
        ))
        print(f'  Tiling {len(train_files)} train chips...')
        done, skip, fail = tile_chips_from_vrt(
            train_files, vrt_path, LOCAL_CHIPS, 'S1Weak'
        )
        print(f'  Train -> done={done}  cached={skip}  failed={fail}')
        total_done += done

    # Tile val chips
    if 'val' in splits:
        val_files = sorted(glob.glob(
            f'{BASE}/val/S1Hand/{country}_*_S1Hand.tif'
        ))
        print(f'  Tiling {len(val_files)} val chips...')
        done, skip, fail = tile_chips_from_vrt(
            val_files, vrt_path, LOCAL_CHIPS, 'S1Hand'
        )
        print(f'  Val   -> done={done}  cached={skip}  failed={fail}')
        total_done += done

    # Clean local tmp (free space for next country)
    for f in local_tiles + [vrt_path, tile_list_path]:
        if os.path.exists(f): os.remove(f)
    print(f'  Local tmp cleaned')

    # Delete Drive mosaics for this country (free Drive space)
    print(f'  Deleting Drive mosaics for {country}...')
    for t in drive_tiles:
        if os.path.exists(t): os.remove(t)
    print(f'  Drive freed ✓')

# ── PHASE 2: Copy chips from local → Drive ────────────────────
print(f'\n{"=" * 55}')
print('PHASE 2: Copying chips to Drive')
print('=' * 55)

local_chip_files = sorted(glob.glob(f'{LOCAL_CHIPS}/*.tif'))
local_size_gb    = sum(os.path.getsize(f) for f in local_chip_files) / 1e9
print(f'Local chips : {len(local_chip_files)} files')
print(f'Local size  : {local_size_gb:.2f} GB')
print(f'Copying to  : {DRIVE_CHIPS}')

for i, f in enumerate(tqdm(local_chip_files)):
    dst = f'{DRIVE_CHIPS}/{os.path.basename(f)}'
    if not os.path.exists(dst):
        shutil.copy2(f, dst)

# ── Final verification ────────────────────────────────────────
print(f'\n{"=" * 55}')
print('VERIFICATION')
print('=' * 55)

drive_chips = glob.glob(f'{DRIVE_CHIPS}/*.tif')
drive_gb    = sum(os.path.getsize(f) for f in drive_chips) / 1e9

print(f'Total chips on Drive : {len(drive_chips)}')
print(f'Expected             : 3185')
print(f'Size on Drive        : {drive_gb:.2f} GB')
print()

all_ok = True
for country in COUNTRIES:
    n   = len(glob.glob(f'{DRIVE_CHIPS}/{country}_*_DEM.tif'))
    exp = len(glob.glob(f'{BASE}/train/S1Weak/{country}_*_S1Weak.tif')) + \
          len(glob.glob(f'{BASE}/val/S1Hand/{country}_*_S1Hand.tif'))
    ok  = '✓' if n == exp else '✗'
    if n != exp: all_ok = False
    print(f'  {country:<12}: {n}/{exp}  {ok}')

print()

# Spot check band count
import rasterio
sample = glob.glob(f'{DRIVE_CHIPS}/India_*_DEM.tif')
if sample:
    with rasterio.open(sample[0]) as src:
        print(f'Band count : {src.count} (expected 4)')
        print(f'Dtype      : {src.dtypes[0]} (expected float32)')
        print(f'Shape      : {src.height}x{src.width} (expected 512x512)')
        ok = src.count == 4 and src.dtypes[0] == 'float32'
        print(f'Band check : {"✓" if ok else "✗"}')

print()
print('Band order: 1=Elevation  2=Slope  3=TWI  4=HAND')
print()
if all_ok:
    print('All verified ✓  Ready for Step 3 (temporal export).')
else:
    print('Some chips missing — re-run cell, it will resume from where it left off.')

Reading package lists...
Building dependency tree...
Reading state information...
gdal-bin is already the newest version (3.8.4+dfsg-1~jammy0).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
PHASE 1: Tiling chips to Colab local disk

India...
  Found 4 mosaic tiles
  Copying to local disk...
  Local size: 9.5 GB
  VRT built successfully
  Tiling 467 train chips...
  Train -> done=467  cached=0  failed=0
  Tiling 68 val chips...
  Val   -> done=68  cached=0  failed=0
  Local tmp cleaned
  Deleting Drive mosaics for India...
  Drive freed ✓

Pakistan...
  Found 6 mosaic tiles
  Copying to local disk...
  Local size: 11.3 GB
  VRT built successfully
  Tiling 249 train chips...
  Train -> done=144  cached=0  failed=105
  Tiling 28 val chips...
  Val   -> done=15  cached=0  failed=13
  Local tmp cleaned
  Deleting Drive mosaics for Pakistan...
  Drive freed ✓

Sri-Lanka...
  Found 2 mosaic tiles
  Copying to local disk...
  Local size: 0.9 GB
  VRT built successfully
  Tilin

100%|██████████| 3067/3067 [03:48<00:00, 13.40it/s]



VERIFICATION
Total chips on Drive : 3067
Expected             : 3185
Size on Drive        : 9.85 GB

  India       : 535/535  ✓
  Pakistan    : 159/277  ✗
  Sri-Lanka   : 232/232  ✓
  Mekong      : 1383/1383  ✓
  Bolivia     : 224/224  ✓
  Colombia    : 534/534  ✓

Band count : 4 (expected 4)
Dtype      : float32 (expected float32)
Shape      : 512x512 (expected 512x512)
Band check : ✓

Band order: 1=Elevation  2=Slope  3=TWI  4=HAND

Some chips missing — re-run cell, it will resume from where it left off.


In [ ]:
import glob, rasterio

BASE = '/content/drive/MyDrive/FloodProject/data/sen1floods11'

for split, suffix in [('train/S1Weak', 'S1Weak'), ('val/S1Hand', 'S1Hand')]:
    files = sorted(glob.glob(f'{BASE}/{split}/Pakistan_*_{suffix}.tif'))
    if not files:
        continue
    minx, miny, maxx, maxy = 999, 999, -999, -999
    for f in files:
        with rasterio.open(f) as src:
            b = src.bounds
            minx = min(minx, b.left)
            miny = min(miny, b.bottom)
            maxx = max(maxx, b.right)
            maxy = max(maxy, b.top)
    print(f'{split}:')
    print(f'  minx={minx:.4f}  miny={miny:.4f}')
    print(f'  maxx={maxx:.4f}  maxy={maxy:.4f}')

train/S1Weak:
  minx=70.2324  miny=29.1140
  maxx=72.3022  maxy=33.8514
val/S1Hand:
  minx=70.4164  miny=29.2060
  maxx=72.3482  maxy=33.8054


In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project='replace GEE project ID with your own')

# ── DEM stack (same as before) ────────────────────────────────
dem   = ee.ImageCollection('COPERNICUS/DEM/GLO30').select('DEM').mosaic()
slope = ee.Terrain.slope(dem).rename('Slope')

slope_rad      = slope.multiply(3.14159265 / 180.0)
tan_slope      = slope_rad.tan()
tan_slope_safe = tan_slope.where(tan_slope.lt(0.001), 0.001)
flow_acc       = dem.reduceNeighborhood(
    reducer = ee.Reducer.sum(),
    kernel  = ee.Kernel.circle(radius=5, units='pixels')
)
twi       = flow_acc.divide(tan_slope_safe).log().rename('TWI')
focal_min = dem.focalMin(radius=50, units='pixels')
hand      = dem.subtract(focal_min).rename('HAND')

dem_stack = dem.rename('Elevation') \
               .addBands(slope) \
               .addBands(twi) \
               .addBands(hand) \
               .toFloat()

# ── Pakistan only — tight bbox around actual chip locations ───
region = ee.Geometry.Rectangle([70.0, 29.0, 73.0, 34.2], 'EPSG:4326', False)

task = ee.batch.Export.image.toDrive(
    image          = dem_stack.clip(region),
    description    = 'Pakistan_DEM_mosaic_v2',
    folder         = 'pakistan_dem_fix',
    fileNamePrefix = 'Pakistan_DEM_mosaic_v2',
    region         = region,
    scale          = 30,
    crs            = 'EPSG:4326',
    fileFormat     = 'GeoTIFF',
    maxPixels      = int(1e13)
)
task.start()
print(f'Pakistan DEM re-export submitted: {task.id}')
print('Monitor at https://code.earthengine.google.com → Tasks tab')
print('Tight bbox: [70.0, 29.0, 73.0, 34.2]')
print('Expected size: much smaller than before (~2-3 GB)')

Pakistan DEM re-export submitted: ME42SYTUJD5GEU7JPNZ6WZTD
Monitor at https://code.earthengine.google.com → Tasks tab
Tight bbox: [70.0, 29.0, 73.0, 34.2]
Expected size: much smaller than before (~2-3 GB)


In [ ]:
import os, glob, shutil, subprocess
import numpy as np
import rasterio
from rasterio.windows import from_bounds
from rasterio.enums import Resampling
from tqdm import tqdm

BASE        = '/content/drive/MyDrive/FloodProject/data/sen1floods11'
PAK_MOSAIC  = '/content/drive/MyDrive/pakistan_dem_fix'
LOCAL_TMP   = '/content/dem_tmp'
DRIVE_CHIPS = f'{BASE}/dem/chips'

os.makedirs(LOCAL_TMP,   exist_ok=True)
os.makedirs(DRIVE_CHIPS, exist_ok=True)

def tile_from_vrt(s1_files, vrt_path, out_dir, suffix):
    done, skipped, failed = 0, 0, 0
    with rasterio.open(vrt_path) as vrt:
        for s1_path in tqdm(s1_files):
            chip_id  = os.path.basename(s1_path).replace(f'_{suffix}.tif', '')
            out_path = f'{out_dir}/{chip_id}_DEM.tif'
            if os.path.exists(out_path):
                skipped += 1
                continue
            try:
                with rasterio.open(s1_path) as s1:
                    bounds    = s1.bounds
                    out_h     = s1.height
                    out_w     = s1.width
                    transform = s1.transform
                    crs       = s1.crs

                vb = vrt.bounds
                if (bounds.right  < vb.left  or
                    bounds.left   > vb.right or
                    bounds.top    < vb.bottom or
                    bounds.bottom > vb.top):
                    failed += 1
                    continue

                window = from_bounds(
                    bounds.left, bounds.bottom,
                    bounds.right, bounds.top,
                    vrt.transform
                )
                data = vrt.read(
                    window     = window,
                    out_shape  = (vrt.count, out_h, out_w),
                    resampling = Resampling.bilinear
                ).astype(np.float32)
                data = np.where(np.isfinite(data), data, 0.0)

                profile = {
                    'driver'   : 'GTiff',
                    'dtype'    : 'float32',
                    'width'    : out_w,
                    'height'   : out_h,
                    'count'    : vrt.count,
                    'crs'      : crs,
                    'transform': transform,
                    'compress' : 'lzw',
                }
                with rasterio.open(out_path, 'w', **profile) as dst:
                    dst.write(data)
                done += 1
            except Exception as e:
                print(f'  FAILED: {chip_id} — {e}')
                failed += 1
    return done, skipped, failed

# ── Copy mosaic tiles to local ────────────────────────────────
pak_tiles = sorted(glob.glob(f'{PAK_MOSAIC}/Pakistan_DEM_mosaic_v2*.tif'))
print(f'Pakistan mosaic tiles: {len(pak_tiles)}')

local_tiles = []
for t in pak_tiles:
    dst = f'{LOCAL_TMP}/{os.path.basename(t)}'
    shutil.copy2(t, dst)
    local_tiles.append(dst)
print(f'Copied {len(local_tiles)} tiles to local')

# ── Build VRT ─────────────────────────────────────────────────
vrt_path       = f'{LOCAL_TMP}/Pakistan_v2.vrt'
tile_list_path = f'{LOCAL_TMP}/Pakistan_v2_tiles.txt'
with open(tile_list_path, 'w') as f:
    f.write('\n'.join(local_tiles))
subprocess.run(
    ['gdalbuildvrt', '-input_file_list', tile_list_path, vrt_path],
    capture_output=True
)
print('VRT built')

# ── Tile only MISSING chips ───────────────────────────────────
train_files = sorted(glob.glob(f'{BASE}/train/S1Weak/Pakistan_*_S1Weak.tif'))
val_files   = sorted(glob.glob(f'{BASE}/val/S1Hand/Pakistan_*_S1Hand.tif'))

print(f'\nTiling {len(train_files)} train chips (skips existing)...')
done, skip, fail = tile_from_vrt(train_files, vrt_path, DRIVE_CHIPS, 'S1Weak')
print(f'Train -> done={done}  cached={skip}  failed={fail}')

print(f'\nTiling {len(val_files)} val chips (skips existing)...')
done, skip, fail = tile_from_vrt(val_files, vrt_path, DRIVE_CHIPS, 'S1Hand')
print(f'Val   -> done={done}  cached={skip}  failed={fail}')

# ── Cleanup ───────────────────────────────────────────────────
for f in local_tiles + [vrt_path, tile_list_path]:
    if os.path.exists(f): os.remove(f)
for t in pak_tiles:
    if os.path.exists(t): os.remove(t)
print('\nCleaned up.')

# ── Final Pakistan count ──────────────────────────────────────
pak_chips = glob.glob(f'{DRIVE_CHIPS}/Pakistan_*_DEM.tif')
print(f'\nPakistan DEM chips: {len(pak_chips)}/277')
print(f'Total DEM chips   : {len(glob.glob(DRIVE_CHIPS+"/*.tif"))}/3185')

Pakistan mosaic tiles: 2
Copied 2 tiles to local
VRT built

Tiling 249 train chips (skips existing)...


100%|██████████| 249/249 [02:05<00:00,  1.98it/s]


Train -> done=105  cached=144  failed=0

Tiling 28 val chips (skips existing)...


100%|██████████| 28/28 [00:07<00:00,  3.52it/s]


Val   -> done=13  cached=15  failed=0

Cleaned up.

Pakistan DEM chips: 277/277
Total DEM chips   : 3185/3185


In [ ]:
import os, glob
import numpy as np
import rasterio

BASE        = '/content/drive/MyDrive/FloodProject/data/sen1floods11'
DRIVE_CHIPS = f'{BASE}/dem/chips'

COUNTRIES = {
    'India':     ['train', 'val'],
    'Pakistan':  ['train', 'val'],
    'Sri-Lanka': ['train', 'val'],
    'Mekong':    ['train', 'val'],
    'Bolivia':   ['train'],
    'Colombia':  ['train'],
}

print('=' * 55)
print('FINAL DEM VERIFICATION')
print('=' * 55)

all_ok = True
for country, splits in COUNTRIES.items():
    dem   = len(glob.glob(f'{DRIVE_CHIPS}/{country}_*_DEM.tif'))
    train = len(glob.glob(f'{BASE}/train/S1Weak/{country}_*_S1Weak.tif'))
    val   = len(glob.glob(f'{BASE}/val/S1Hand/{country}_*_S1Hand.tif')) \
            if 'val' in splits else 0
    exp   = train + val
    ok    = '✓' if dem == exp else '✗'
    if dem != exp: all_ok = False
    print(f'  {country:<12}: {dem}/{exp}  {ok}')

total_dem = len(glob.glob(f'{DRIVE_CHIPS}/*.tif'))
print(f'  {"TOTAL":<12}: {total_dem}/3185  {"✓" if total_dem==3185 else "✗"}')

print()
print('=' * 55)
print('STORAGE SUMMARY')
print('=' * 55)
folders = {
    'train/S1Weak'          : f'{BASE}/train/S1Weak',
    'train/S2Weak'          : f'{BASE}/train/S2Weak',
    'train/S2IndexLabelWeak': f'{BASE}/train/S2IndexLabelWeak',
    'val/S1Hand'            : f'{BASE}/val/S1Hand',
    'val/S2Hand'            : f'{BASE}/val/S2Hand',
    'val/LabelHand'         : f'{BASE}/val/LabelHand',
    'dem/chips'             : DRIVE_CHIPS,
}
total_gb = 0
for name, folder in folders.items():
    files = glob.glob(f'{folder}/*.tif')
    gb    = sum(os.path.getsize(f) for f in files) / 1e9
    total_gb += gb
    print(f'  {name:<25}: {len(files):>5} files  {gb:>6.2f} GB')
print(f'  {"TOTAL":<25}: {total_gb:>6.2f} GB')

print()
if all_ok:
    print('ALL CHECKS PASSED ✓')
    print('DEM step complete. Ready for Step 3 — Temporal Export.')
else:
    print('ISSUES FOUND ✗')

FINAL DEM VERIFICATION
  India       : 535/535  ✓
  Pakistan    : 277/277  ✓
  Sri-Lanka   : 232/232  ✓
  Mekong      : 1383/1383  ✓
  Bolivia     : 224/224  ✓
  Colombia    : 534/534  ✓
  TOTAL       : 3185/3185  ✓

STORAGE SUMMARY
  train/S1Weak             :  3017 files    5.16 GB
  train/S2Weak             :  3017 files   17.83 GB
  train/S2IndexLabelWeak   :  3017 files    0.03 GB
  val/S1Hand               :   168 files    0.28 GB
  val/S2Hand               :   168 files    0.39 GB
  val/LabelHand            :   168 files    0.00 GB
  dem/chips                :  3185 files   10.20 GB
  TOTAL                    :  33.89 GB

ALL CHECKS PASSED ✓
DEM step complete. Ready for Step 3 — Temporal Export.
